# GraphQL-Style Operations on Topologic Graphs

This notebook demonstrates how to perform graph query operations on topologic_fast graphs using a GQL-inspired approach.

**Note:** Native GQL integration is not yet implemented in topologic_fast. This notebook shows:
1. How to create and analyze topological graphs
2. A Python-based implementation of graph queries
3. Pattern matching, filtering, and aggregation operations

## What is GQL?

GQL (Graph Query Language) is an emerging standard for querying property graphs. It combines concepts from:
- **Cypher** (Neo4j)
- **PGQL** (Oracle)
- **GSQL** (TigerGraph)
- **G-CORE** (Academic standard)

Key operations include:
- `MATCH` - Pattern matching
- `WHERE` - Filtering
- `RETURN` - Projection
- `CREATE` / `MERGE` - Mutation
- `SET` / `DELETE` - Updates

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from collections import Counter
import re
from typing import List, Dict, Any, Optional, Callable
from dataclasses import dataclass

## 1. Create a Sample Building Graph

Let's create a CellComplex representing a building and generate its connectivity graph.

In [ ]:
# Create a simple 2-floor building
floor_height = 3.0
rooms = []
room_metadata = []

def add_room(x, y, z, w, l, h, name, room_type, floor):
    room = tf.Cell.Box(x, y, z, w, l, h)
    rooms.append(room)
    room_metadata.append({
        'name': name,
        'type': room_type,
        'floor': floor,
        'area': w * l
    })
    return room

# Ground floor
add_room(0, 0, 0, 5, 5, 3, 'Lobby', 'lobby', 0)
add_room(5, 0, 0, 5, 5, 3, 'Reception', 'reception', 0)
add_room(10, 0, 0, 5, 5, 3, 'Meeting_A', 'meeting', 0)
add_room(0, 5, 0, 5, 5, 3, 'Office_1', 'office', 0)
add_room(5, 5, 0, 5, 5, 3, 'Office_2', 'office', 0)
add_room(10, 5, 0, 5, 5, 3, 'Kitchen', 'kitchen', 0)

# First floor
add_room(0, 0, 3, 5, 5, 3, 'Conference_A', 'conference', 1)
add_room(5, 0, 3, 5, 5, 3, 'Conference_B', 'conference', 1)
add_room(10, 0, 3, 5, 5, 3, 'Meeting_B', 'meeting', 1)
add_room(0, 5, 3, 5, 5, 3, 'Office_3', 'office', 1)
add_room(5, 5, 3, 5, 5, 3, 'Office_4', 'office', 1)
add_room(10, 5, 3, 5, 5, 3, 'Break_Room', 'break', 1)

# Create CellComplex and graph
building = tf.CellComplex.ByCells(rooms)
graph = tf.Graph.ByTopology(building)

print(f"Building: {building.NumCells()} rooms")
print(f"Graph: {graph.Order()} vertices, {graph.Size()} edges")

## 2. Create a Graph Query Wrapper

We'll create a simple query interface that mimics GQL operations.

In [ ]:
@dataclass
class GraphNode:
    """Wrapper for a graph vertex with metadata."""
    id: int
    vertex: Any  # tf.Vertex
    properties: Dict[str, Any]
    
    def __getitem__(self, key):
        return self.properties.get(key)
    
    def get(self, key, default=None):
        return self.properties.get(key, default)


@dataclass
class GraphEdge:
    """Wrapper for a graph edge."""
    id: int
    edge: Any  # tf.Edge
    from_id: int
    to_id: int
    properties: Dict[str, Any]
    
    def __getitem__(self, key):
        return self.properties.get(key)


class TopologicGraph:
    """A queryable wrapper around a topologic_fast Graph."""
    
    def __init__(self, tf_graph, vertex_metadata: List[Dict] = None):
        self.tf_graph = tf_graph
        self.nodes: List[GraphNode] = []
        self.edges: List[GraphEdge] = []
        
        # Index vertices
        vertices = tf_graph.Vertices()
        self._coord_to_id = {}
        
        for i, v in enumerate(vertices):
            coords = v.Coordinates()
            coord_key = (round(coords[0], 3), round(coords[1], 3), round(coords[2], 3))
            self._coord_to_id[coord_key] = i
            
            props = vertex_metadata[i] if vertex_metadata and i < len(vertex_metadata) else {}
            props['x'] = coords[0]
            props['y'] = coords[1]
            props['z'] = coords[2]
            props['id'] = i
            
            self.nodes.append(GraphNode(id=i, vertex=v, properties=props))
        
        # Index edges
        edges = tf_graph.Edges()
        for i, e in enumerate(edges):
            edge_verts = e.Vertices()
            if len(edge_verts) == 2:
                from_coords = edge_verts[0].Coordinates()
                to_coords = edge_verts[1].Coordinates()
                
                from_key = (round(from_coords[0], 3), round(from_coords[1], 3), round(from_coords[2], 3))
                to_key = (round(to_coords[0], 3), round(to_coords[1], 3), round(to_coords[2], 3))
                
                from_id = self._coord_to_id.get(from_key, -1)
                to_id = self._coord_to_id.get(to_key, -1)
                
                if from_id >= 0 and to_id >= 0:
                    # Determine edge type
                    if abs(from_coords[2] - to_coords[2]) > 0.1:
                        rel_type = 'VERTICAL'
                    else:
                        rel_type = 'HORIZONTAL'
                    
                    self.edges.append(GraphEdge(
                        id=i, edge=e,
                        from_id=from_id, to_id=to_id,
                        properties={'type': rel_type}
                    ))
        
        # Build adjacency
        self._adjacency = {i: [] for i in range(len(self.nodes))}
        for e in self.edges:
            self._adjacency[e.from_id].append((e.to_id, e))
            self._adjacency[e.to_id].append((e.from_id, e))  # Undirected
    
    def match_nodes(self, 
                    where: Callable[[GraphNode], bool] = None,
                    category: str = None) -> List[GraphNode]:
        """MATCH (n) WHERE ... pattern."""
        results = []
        for node in self.nodes:
            if category and node.get('type') != category:
                continue
            if where and not where(node):
                continue
            results.append(node)
        return results
    
    def match_edges(self,
                    where: Callable[[GraphEdge], bool] = None,
                    rel_type: str = None) -> List[GraphEdge]:
        """MATCH ()-[r]->() WHERE ... pattern."""
        results = []
        for edge in self.edges:
            if rel_type and edge.get('type') != rel_type:
                continue
            if where and not where(edge):
                continue
            results.append(edge)
        return results
    
    def match_pattern(self,
                      node_a_filter: Callable[[GraphNode], bool] = None,
                      rel_filter: Callable[[GraphEdge], bool] = None,
                      node_b_filter: Callable[[GraphNode], bool] = None) -> List[Dict]:
        """MATCH (a)-[r]-(b) WHERE ... pattern."""
        results = []
        for edge in self.edges:
            node_a = self.nodes[edge.from_id]
            node_b = self.nodes[edge.to_id]
            
            if node_a_filter and not node_a_filter(node_a):
                continue
            if rel_filter and not rel_filter(edge):
                continue
            if node_b_filter and not node_b_filter(node_b):
                continue
            
            results.append({'a': node_a, 'r': edge, 'b': node_b})
        return results
    
    def adjacent_nodes(self, node: GraphNode) -> List[GraphNode]:
        """Get adjacent nodes."""
        return [self.nodes[neighbor_id] for neighbor_id, _ in self._adjacency[node.id]]
    
    def shortest_path(self, from_node: GraphNode, to_node: GraphNode) -> List[GraphNode]:
        """Find shortest path using BFS."""
        from collections import deque
        
        if from_node.id == to_node.id:
            return [from_node]
        
        visited = {from_node.id}
        queue = deque([(from_node.id, [from_node])])
        
        while queue:
            current_id, path = queue.popleft()
            
            for neighbor_id, _ in self._adjacency[current_id]:
                if neighbor_id == to_node.id:
                    return path + [self.nodes[neighbor_id]]
                
                if neighbor_id not in visited:
                    visited.add(neighbor_id)
                    queue.append((neighbor_id, path + [self.nodes[neighbor_id]]))
        
        return []  # No path found
    
    def count(self, items: List) -> int:
        """COUNT(*) aggregation."""
        return len(items)
    
    def group_by(self, items: List[Dict], key: str) -> Dict[str, List]:
        """GROUP BY operation."""
        groups = {}
        for item in items:
            # Handle both dict and GraphNode
            if isinstance(item, dict):
                val = item.get(key)
            else:
                val = item.get(key)
            
            if val not in groups:
                groups[val] = []
            groups[val].append(item)
        return groups

# Create queryable graph
qgraph = TopologicGraph(graph, room_metadata)
print(f"TopologicGraph created with {len(qgraph.nodes)} nodes and {len(qgraph.edges)} edges")

## 3. Execute Graph Queries

Now let's run various GQL-style queries on the graph.

In [ ]:
# Query 1: MATCH (n) RETURN COUNT(*) AS total
print("Query 1: Count all nodes")
print(f"  -- GQL: MATCH (n) RETURN COUNT(*) AS total")
all_nodes = qgraph.match_nodes()
print(f"  Result: {qgraph.count(all_nodes)} nodes")

In [ ]:
# Query 2: MATCH (n) WHERE n.type = 'office' RETURN n
print("Query 2: Find all offices")
print(f"  -- GQL: MATCH (n) WHERE n.type = 'office' RETURN n.name")
offices = qgraph.match_nodes(where=lambda n: n.get('type') == 'office')
for office in offices:
    print(f"  - {office['name']} (floor {office['floor']})")

In [ ]:
# Query 3: MATCH (n) WHERE n.floor = 0 RETURN DISTINCT n.type, COUNT(*)
print("Query 3: Room types on ground floor")
print(f"  -- GQL: MATCH (n) WHERE n.floor = 0 RETURN DISTINCT n.type, COUNT(*)")
ground_floor = qgraph.match_nodes(where=lambda n: n.get('floor') == 0)
type_counts = Counter(n.get('type') for n in ground_floor)
for room_type, count in type_counts.items():
    print(f"  - {room_type}: {count}")

In [ ]:
# Query 4: MATCH (a)-[r:VERTICAL]-(b) RETURN a.name, b.name
print("Query 4: Find vertically connected rooms (between floors)")
print(f"  -- GQL: MATCH (a)-[r:VERTICAL]-(b) RETURN a.name, b.name")
vertical = qgraph.match_pattern(
    rel_filter=lambda r: r.get('type') == 'VERTICAL'
)
for match in vertical:
    print(f"  - {match['a']['name']} <-> {match['b']['name']}")

In [ ]:
# Query 5: MATCH (a {type: 'office'})-[r]-(b {type: 'conference'})
print("Query 5: Find offices adjacent to conference rooms")
print(f"  -- GQL: MATCH (a:office)-[r]-(b:conference) RETURN a.name, b.name")
office_conf = qgraph.match_pattern(
    node_a_filter=lambda n: n.get('type') == 'office',
    node_b_filter=lambda n: n.get('type') == 'conference'
)
if office_conf:
    for match in office_conf:
        print(f"  - {match['a']['name']} <-> {match['b']['name']}")
else:
    print("  No direct connections found")

In [ ]:
# Query 6: Shortest path
print("Query 6: Shortest path from Lobby to Break_Room")
print(f"  -- GQL: MATCH path = shortestPath((a {{name: 'Lobby'}})-[*]-(b {{name: 'Break_Room'}})) RETURN path")

lobby = qgraph.match_nodes(where=lambda n: n.get('name') == 'Lobby')[0]
break_room = qgraph.match_nodes(where=lambda n: n.get('name') == 'Break_Room')[0]

path = qgraph.shortest_path(lobby, break_room)
if path:
    path_names = [n['name'] for n in path]
    print(f"  Path length: {len(path) - 1} steps")
    print(f"  Route: {' -> '.join(path_names)}")

In [ ]:
# Query 7: Node degree (connection count)
print("Query 7: Rooms with most connections")
print(f"  -- GQL: MATCH (n)-[r]-() RETURN n.name, COUNT(r) AS connections ORDER BY connections DESC")

connection_counts = []
for node in qgraph.nodes:
    adjacent = qgraph.adjacent_nodes(node)
    connection_counts.append((node['name'], len(adjacent)))

connection_counts.sort(key=lambda x: -x[1])
for name, count in connection_counts[:5]:
    print(f"  - {name}: {count} connections")

In [ ]:
# Query 8: Find all neighbors of a room
print("Query 8: All rooms adjacent to Office_2")
print(f"  -- GQL: MATCH (n {{name: 'Office_2'}})-[r]-(neighbor) RETURN neighbor.name")

office_2 = qgraph.match_nodes(where=lambda n: n.get('name') == 'Office_2')[0]
neighbors = qgraph.adjacent_nodes(office_2)
for neighbor in neighbors:
    print(f"  - {neighbor['name']} ({neighbor['type']}, floor {neighbor['floor']})")

## 4. Visualize Query Results

In [ ]:
def visualize_query_result(qgraph, highlighted_nodes=None, highlighted_edges=None, title="Graph"):
    """Visualize graph with highlighted query results."""
    fig = go.Figure()
    
    # Draw all edges
    for edge in qgraph.edges:
        node_a = qgraph.nodes[edge.from_id]
        node_b = qgraph.nodes[edge.to_id]
        
        is_highlighted = highlighted_edges and edge in highlighted_edges
        color = 'red' if is_highlighted else ('blue' if edge['type'] == 'HORIZONTAL' else 'orange')
        width = 5 if is_highlighted else 2
        
        fig.add_trace(go.Scatter3d(
            x=[node_a['x'], node_b['x']],
            y=[node_a['y'], node_b['y']],
            z=[node_a['z'], node_b['z']],
            mode='lines',
            line=dict(color=color, width=width),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw nodes
    highlighted_ids = set(n.id for n in (highlighted_nodes or []))
    
    for node in qgraph.nodes:
        is_highlighted = node.id in highlighted_ids
        color = 'red' if is_highlighted else 'lightblue'
        size = 15 if is_highlighted else 10
        
        fig.add_trace(go.Scatter3d(
            x=[node['x']],
            y=[node['y']],
            z=[node['z']],
            mode='markers+text',
            marker=dict(size=size, color=color, line=dict(color='black', width=1)),
            text=[node['name']],
            textposition='top center',
            name=node['name'],
            hovertext=f"{node['name']}\nType: {node['type']}\nFloor: {node['floor']}",
            hoverinfo='text',
            showlegend=False
        ))
    
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data'),
        width=900,
        height=600
    )
    
    return fig

# Visualize all offices highlighted
offices = qgraph.match_nodes(where=lambda n: n.get('type') == 'office')
fig = visualize_query_result(qgraph, highlighted_nodes=offices, title="MATCH (n:office) - Highlighted")
fig.show()

In [ ]:
# Visualize shortest path
path = qgraph.shortest_path(lobby, break_room)
fig = visualize_query_result(qgraph, highlighted_nodes=path, title="Shortest Path: Lobby -> Break_Room")
fig.show()

## 5. Mutation Operations (Simulated)

GQL also supports mutation operations. Here's how they would work.

In [ ]:
# SET operation: Update node properties
print("SET operation: Update Office_1 to 'executive' type")
print(f"  -- GQL: MATCH (n {{name: 'Office_1'}}) SET n.type = 'executive'")

# Find and update
office_1 = qgraph.match_nodes(where=lambda n: n.get('name') == 'Office_1')[0]
old_type = office_1.properties['type']
office_1.properties['type'] = 'executive'
print(f"  Updated: {office_1['name']} type changed from '{old_type}' to '{office_1['type']}'")

# Revert for consistency
office_1.properties['type'] = old_type

In [ ]:
# CREATE operation: Add a new edge (relationship)
print("\nCREATE operation: Add a direct connection between Lobby and Kitchen")
print(f"  -- GQL: MATCH (a {{name: 'Lobby'}}), (b {{name: 'Kitchen'}})")
print(f"         CREATE (a)-[:CORRIDOR]->(b)")

lobby = qgraph.match_nodes(where=lambda n: n.get('name') == 'Lobby')[0]
kitchen = qgraph.match_nodes(where=lambda n: n.get('name') == 'Kitchen')[0]

# Check if connection exists
existing = qgraph.match_pattern(
    node_a_filter=lambda n: n.get('name') == 'Lobby',
    node_b_filter=lambda n: n.get('name') == 'Kitchen'
)

if existing:
    print(f"  Connection already exists")
else:
    print(f"  Would create new CORRIDOR relationship")
    # In a real implementation, we would add the edge to the graph

## 6. Advanced Aggregation Queries

In [ ]:
# Group by floor and type
print("Aggregation: Room count by floor and type")
print(f"  -- GQL: MATCH (n) RETURN n.floor, n.type, COUNT(*) GROUP BY n.floor, n.type")

for floor in [0, 1]:
    floor_rooms = qgraph.match_nodes(where=lambda n: n.get('floor') == floor)
    type_groups = qgraph.group_by(floor_rooms, 'type')
    
    print(f"\n  Floor {floor}:")
    for room_type, rooms in type_groups.items():
        print(f"    - {room_type}: {len(rooms)}")

In [ ]:
# Graph statistics
print("Graph Statistics:")
print(f"  Total nodes: {len(qgraph.nodes)}")
print(f"  Total edges: {len(qgraph.edges)}")

horizontal = qgraph.match_edges(rel_type='HORIZONTAL')
vertical = qgraph.match_edges(rel_type='VERTICAL')
print(f"  Horizontal edges: {len(horizontal)}")
print(f"  Vertical edges: {len(vertical)}")

# Degree distribution
degrees = [len(qgraph.adjacent_nodes(n)) for n in qgraph.nodes]
print(f"  Average degree: {sum(degrees)/len(degrees):.2f}")
print(f"  Max degree: {max(degrees)}")
print(f"  Min degree: {min(degrees)}")

## 7. Example Cypher/GQL Queries Reference

Here are common GQL patterns that would be supported in a full implementation.

In [ ]:
gql_examples = """
-- Basic Pattern Matching --

-- All nodes
MATCH (n) RETURN n

-- Nodes by type
MATCH (n:Office) RETURN n.name, n.floor

-- Nodes with property filter
MATCH (n) WHERE n.floor = 0 AND n.area > 20 RETURN n

-- All edges/relationships
MATCH (a)-[r]->(b) RETURN a.name, type(r), b.name

-- Edges by type
MATCH (a)-[r:HORIZONTAL]->(b) RETURN a.name, b.name

-- Pattern with both node and edge filters
MATCH (a:Office)-[r:HORIZONTAL]-(b:Office)
WHERE a.floor = b.floor
RETURN a.name, b.name


-- Aggregations --

-- Count by type
MATCH (n) RETURN n.type, COUNT(*) AS count ORDER BY count DESC

-- Group by floor
MATCH (n) RETURN n.floor, COLLECT(n.name) AS rooms

-- Node degree
MATCH (n)-[r]-() RETURN n.name, COUNT(r) AS connections


-- Path Queries --

-- Shortest path
MATCH path = shortestPath((a {name: 'Lobby'})-[*]-(b {name: 'Office_4'}))
RETURN path

-- All paths up to length 3
MATCH path = (a {name: 'Lobby'})-[*1..3]-(b)
RETURN b.name, length(path)

-- K-hop neighbors
MATCH (start {name: 'Office_2'})-[*1..2]-(neighbor)
RETURN DISTINCT neighbor.name


-- Mutations --

-- Create node
CREATE (n:Room {name: 'Stairwell', type: 'circulation', floor: 0})

-- Create relationship
MATCH (a {name: 'Lobby'}), (b {name: 'Stairwell'})
CREATE (a)-[:CONNECTS_TO]->(b)

-- Update property
MATCH (n {name: 'Office_1'}) SET n.type = 'executive'

-- Delete relationship
MATCH (a)-[r:HORIZONTAL]-(b)
WHERE a.name = 'Lobby' AND b.name = 'Reception'
DELETE r

-- Merge (create if not exists)
MERGE (n:Room {name: 'Corridor_1'})
ON CREATE SET n.type = 'circulation'
ON MATCH SET n.last_accessed = timestamp()
"""

print(gql_examples)

## Summary

This notebook demonstrated:

1. **Graph Creation** - Building a CellComplex and its dual graph
2. **Query Wrapper** - A Python implementation of GQL-style operations
3. **Pattern Matching** - MATCH operations on nodes and edges
4. **Filtering** - WHERE clauses with lambda predicates
5. **Aggregation** - COUNT, GROUP BY operations
6. **Path Queries** - Shortest path algorithms
7. **Visualization** - Highlighting query results

### Features Not Yet Implemented in topologic_fast:

- `tf.GQL` - Native GQL query engine
- `tf.GQL.Query()` - Execute GQL string queries
- `tf.Graph.Match()` - Pattern matching on graphs
- `tf.Dictionary` - Attaching metadata to topologies

### GQL Resources:

- [GQL Standard (ISO)](https://www.gqlstandards.org/)
- [openCypher](https://opencypher.org/)
- [Neo4j Cypher Manual](https://neo4j.com/docs/cypher-manual/current/)